In [1]:
import ROOT 

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import os

In [117]:
def reject_outliers_zscore(data, threshold=3):
    # Calculate mean and standard deviation
    data = np.array(data)
    mu = np.mean(data)
    std = np.std(data)
    # Calculate absolute Z-scores for each data point
    z_scores = np.abs((data - mu) / std)
    # Filter the data for non-outliers (where z-score is less than threshold)
    filtered_data = data[z_scores < threshold]
    return filtered_data

def MakeChain(path,header,tree_name):
    chain = ROOT.TChain(tree_name)
    for entry in os.listdir(path):
        if header not in entry: continue
        chain.Add(os.path.join(path, entry))
    return chain

def FillEHistogram(hs,chain):
    for event in chain:
        for i,h in enumerate(hs):
            branch_name = f"Waveform00{i}"
            wfm = getattr(event, branch_name)
            amp = np.max(wfm) - np.mean(reject_outliers_zscore(wfm))#np.mean(wfm)#
            h.Fill(amp)

def QuickPlotHisto(h):
    c = ROOT.TCanvas()
    h.Draw("HIst")
    c.Draw()

In [147]:
file_path = "/Users/ebrandani/CERES_UV_NTD/"
file_header = "CERES_TeO2_UV_R_10V_100us_NTD1_0V_NTD2_10V"
chain = MakeChain(file_path,file_header,"data_tree")

ch1 = ROOT.TH1F("ch1","ch1",100,0.2,.6)
ch2 = ROOT.TH1F("ch2","ch2",100,0.6,2)
FillEHistogram([ch1,ch2],chain)

Warning in <TROOT::Append>: Replacing existing TH1: ch1 (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: ch2 (Potential memory leak).


In [158]:
c = ROOT.TCanvas()
fit1 = ch1.Fit("gaus","QS")
ch1.Draw("HIst")
c.Draw()

In [159]:
c = ROOT.TCanvas()
fit2 = ch2.Fit("gaus","QS")
ch2.Draw("HIst")
c.Draw()